# 第5课：损失函数与优化器

**学习目标：**
- 使用 `CrossEntropyLoss` 作为分类损失
- 使用 `optim.SGD` 优化器更新权重
- 理解完整的 PyTorch 训练循环

---

在 NumPy 教程中，我们手写了损失函数、需求函数和权重更新。PyTorch 把这些全部封装好了：
- **损失函数**：`nn.CrossEntropyLoss`
- **优化器**：`torch.optim.SGD`（自动计算梯度并更新权重）

## 5.1 CrossEntropyLoss

分类任务的标准损失函数。它内部已经包含了 Softmax，所以网络输出原始 logits 即可。

**注意：** `CrossEntropyLoss` 接收的是 logits（未归一化的分数），不是概率。

In [ ]:
import torch
import torch.nn as nn

# 模拟网络输出（logits）和真实标签
logits = torch.tensor([[2.0, 1.0],   # 样本1
                        [0.5, 3.0],   # 样本2
                        [1.0, 0.5]])  # 样本3
# shape: (3, 2) — 3个样本，2个类别

labels = torch.tensor([0, 1, 0])  # 真实类别
# shape: (3,) — 每个样本一个类别标签

# 计算损失
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print("损失:", loss.item())

## 5.2 优化器

`optim.SGD` 封装了：
1. 梯度清零：`optimizer.zero_grad()`
2. 反向传播：`loss.backward()`（仍然需要手动调用）
3. 参数更新：`optimizer.step()`

不再需要手动写 `param.data -= lr * param.grad`。

In [ ]:
# 创建简单网络
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

# 创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 模拟一个训练步骤
x = torch.randn(4, 2)        # 4个样本
labels = torch.tensor([0, 1, 1, 0])

# 训练步骤
optimizer.zero_grad()          # 1. 清零梯度
logits = model(x)              # 2. 前向传播
loss = nn.CrossEntropyLoss()(logits, labels)  # 3. 计算损失
loss.backward()                # 4. 反向传播
optimizer.step()               # 5. 更新权重

print("损失:", loss.item())

## 5.3 完整训练循环

标准的 PyTorch 训练循环：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from pytorch.utils import create_data, plot_data

# 数据准备
data = create_data(500)
X = torch.tensor(data[:, :2], dtype=torch.float32)
y = torch.tensor(data[:, 2], dtype=torch.long)

# 模型
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 训练
losses = []
n_epochs = 200

for epoch in range(n_epochs):
    # 前向传播
    logits = model(X)
    loss = criterion(logits, y)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

In [ ]:
# 可视化
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("训练损失曲线")

plt.subplot(1, 2, 2)
model.eval()
with torch.no_grad():
    predictions = torch.argmax(model(X), dim=1).numpy()
result_data = data.copy()
result_data[:, 2] = predictions
plot_data(result_data, "训练后分类结果")

plt.tight_layout()
plt.show()

---

## NumPy vs PyTorch 训练对比

| 步骤 | NumPy 手写 | PyTorch |
| --- | --- | --- |
| 前向传播 | `net.forward(x)` | `model(x)` |
| 损失计算 | `loss_function(pred, real)` | `nn.CrossEntropyLoss()(logits, y)` |
| 反向传播 | `net.backward(real)` | `loss.backward()` |
| 权重更新 | `W += lr * grad` | `optimizer.step()` |
| 梯度清零 | 无（手动管理） | `optimizer.zero_grad()` |

---

## 小结

- `nn.CrossEntropyLoss` = Softmax + 交叉熵，一步到位
- `optim.SGD/Adam` 自动管理梯度更新
- 训练循环：forward → loss → backward → step
- PyTorch 的训练代码比 NumPy 简洁 5 倍以上

**下一课**是综合实战，我们将完成端到端的训练并可视化决策边界。